In [1]:
import pandas as pd
import numpy as np
import keras
from keras import models
from keras import layers
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [2]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load IMDB dataset keeping only the top 1000 most frequent words
vocab_size = 1000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print("Data loaded successfully")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Data loaded successfully
Training samples: 25000
Testing samples: 25000


In [3]:
# help(imdb.get_word_index)

In [4]:
# Word index mapping (word -> integer index)
word_index = imdb.get_word_index()

# Reverse mapping (integer index -> word), needed to decode reviews
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

def decode_review(encoded_review):
    # indices are offset by 3 because 0,1,2 are reserved for padding, start-of-seq, unknown
    return " ".join([reverse_word_index.get(i-3, "?") for i in encoded_review])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [5]:
# for (key, value) in word_index.items():
#   print(key, value)
#   break

In [6]:
# reverse_word_index[1], reverse_word_index[14], reverse_word_index[22], reverse_word_index[16], reverse_word_index[43]

In [7]:
for i in range(5):
    print(f"Review {i+1}")
    print("Label:", "Positive" if y_train[i] == 1 else "Negative")
    print("Encoded review:\n", X_train[i])
    print("\nDecoded review:\n", decode_review(X_train[i]))
    print("-" * 100)

Review 1
Label: Positive
Encoded review:
 [1, 14, 22, 16, 43, 530, 973, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 2, 2, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2, 19, 14, 22, 4, 2, 2, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 2, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2, 2, 16, 480, 66, 2, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 2, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 2, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 2, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 2, 88, 12, 16, 283, 5, 16, 2, 113, 103, 32, 15, 16, 2, 19, 178, 32]

Decoded review:
 ? this film was just brilliant casting ? ? story direction ? really ?

In [8]:
review_lengths = [len(review) for review in X_train]

shortest_len = min(review_lengths)
longest_len = max(review_lengths)
mean_len = int(np.mean(review_lengths))

print("Length of shortest review:", shortest_len)
print("Length of longest review:", longest_len)
print("Average (mean) review length:", mean_len)

Length of shortest review: 11
Length of longest review: 2494
Average (mean) review length: 238


In [9]:
maxlen = 250  # use the average review length computed above

X_train_padded = pad_sequences(X_train, maxlen=maxlen, padding='post', value=0)
X_test_padded = pad_sequences(X_test, maxlen=maxlen, padding='post', value=0)

print("Padding length used (mean review length):", maxlen)
print("Shape of padded training data:", X_train_padded.shape)
print("Shape of padded testing data:", X_test_padded.shape)
print("\nExample padded review (review 1):\n", X_train_padded[0])

Padding length used (mean review length): 250
Shape of padded training data: (25000, 250)
Shape of padded testing data: (25000, 250)

Example padded review (review 1):
 [  1  14  22  16  43 530 973   2   2  65 458   2  66   2   4 173  36 256
   5  25 100  43 838 112  50 670   2   9  35 480 284   5 150   4 172 112
 167   2 336 385  39   4 172   2   2  17 546  38  13 447   4 192  50  16
   6 147   2  19  14  22   4   2   2 469   4  22  71  87  12  16  43 530
  38  76  15  13   2   4  22  17 515  17  12  16 626  18   2   5  62 386
  12   8 316   8 106   5   4   2   2  16 480  66   2  33   4 130  12  16
  38 619   5  25 124  51  36 135  48  25   2  33   6  22  12 215  28  77
  52   5  14 407  16  82   2   8   4 107 117   2  15 256   4   2   7   2
   5 723  36  71  43 530 476  26 400 317  46   7   4   2   2  13 104  88
   4 381  15 297  98  32   2  56  26 141   6 194   2  18   4 226  22  21
 134 476  26 480   5 144  30   2  18  51  36  28 224  92  25 104   4 226
  65  16  38   2  88  12  16

In [10]:
print("Number of training records:", len(X_train))
print("Number of testing records:", len(X_test))

train_pos = np.sum(y_train == 1)
train_neg = np.sum(y_train == 0)
test_pos = np.sum(y_test == 1)
test_neg = np.sum(y_test == 0)

print("\nTraining set -> Positive:", train_pos, " Negative:", train_neg)
print("Testing set  -> Positive:", test_pos, " Negative:", test_neg)

Number of training records: 25000
Number of testing records: 25000

Training set -> Positive: 12500  Negative: 12500
Testing set  -> Positive: 12500  Negative: 12500


In [11]:
from keras.models import Sequential
from keras.layers import Embedding, GlobalAveragePooling1D, Dense
from keras.optimizers import Adam

model = Sequential([
    Embedding(input_dim=1000, output_dim=30, input_length=X_train_padded.shape[1]),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train_padded,
    y_train,
    epochs=100,
    batch_size=16,
    verbose=0
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
test_loss, test_accuracy = model.evaluate(
    X_test_padded,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

Test Loss: 0.9294905662536621
Test Accuracy: 0.8346800208091736
